In [1]:
import json
from pathlib import Path
from datasets import Dataset, DatasetDict
from pff import data_dir
from huggingface_hub import login

login()

In [2]:
def preprocess(studies):
    """
    Clean and normalize a list of study dictionaries by:
      - Flattening nested list structures in "context" and "target".
      - Removing empty covariate entries from individuals.
      - Normalizing dosing fields so they are always lists.

    Each input study is expected to be a dictionary with keys:
      - "context": list of individuals (dicts) or nested lists of individuals
      - "target": list of individuals (dicts) or nested lists of individuals
      - "meta_data": optional dictionary with metadata

    Rules applied:
      1. Flatten any nested lists inside "context"/"target".
      2. For each individual (dict):
         - Remove "covariates" if empty (None, {}, []).
         - Normalize dosing fields ("dosing", "dosing_type",
           "dosing_times", "dosing_name") to always be lists.
      3. Preserve "meta_data".
      4. Return a cleaned list of studies.

    Parameters
    ----------
    studies : list[dict]
        List of study dictionaries.

    Returns
    -------
    list[dict]
        Normalized list of studies.
    """
    fixed = []
    dosing_fields = ["dosing", "dosing_type", "dosing_times", "dosing_name"]

    for s in studies:
        new_s = {"context": [], "target": [], "meta_data": s.get("meta_data", {})}
        for block in ["context", "target"]:
            raw_block = s.get(block, [])
            # flatten: if entries are lists, extend; otherwise, append
            flat_inds = []
            for ind in raw_block:
                if isinstance(ind, list):
                    flat_inds.extend(ind)
                else:
                    flat_inds.append(ind)
            # process individuals
            for ind in flat_inds:
                ind_copy = ind.copy()

                # drop empty covariates
                cov = ind_copy.get("covariates", None)
                if cov is None or cov == {} or cov == []:
                    ind_copy.pop("covariates", None)

                # normalize dosing fields to list
                for f in dosing_fields:
                    if f in ind_copy and not isinstance(ind_copy[f], list):
                        ind_copy[f] = [ind_copy[f]]

                new_s[block].append(ind_copy)
        fixed.append(new_s)
    return fixed



In [4]:
json_path=Path(data_dir) / "preprocessed" / "Theophylline.json"

with json_path.open() as f:
    raw_studies = json.load(f)

In [5]:
flat_studies  = preprocess(raw_studies)

In [1]:
#flat_studies

In [7]:
# Create dataset
dataset = Dataset.from_list(flat_studies)
ds_dict = DatasetDict({"train": dataset})

# Push to hub
ds_dict.push_to_hub("cesarali/Theophylline")


# 5. Usage example (for others)
# from datasets import load_dataset
# ds = load_dataset("your-username/pk-study-json")
# print(ds["train"][0])


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/9.93k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/cesarali/Theophylline/commit/453d704f85cc9234c3b9f9f5898f420f958b9eee', commit_message='Upload dataset', commit_description='', oid='453d704f85cc9234c3b9f9f5898f420f958b9eee', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/cesarali/Theophylline', endpoint='https://huggingface.co', repo_type='dataset', repo_id='cesarali/Theophylline'), pr_revision=None, pr_num=None)

In [18]:
flat_studies[0].keys()

dict_keys(['context', 'target', 'meta_data'])

In [22]:
flat_studies[0]["context"][0]

{'name_id': 'Theophylline_6',
 'observations': [0,
  1.29,
  3.08,
  6.44,
  6.32,
  5.53,
  4.94,
  4.02,
  3.46,
  2.78,
  0.92],
 'observation_times': [0,
  0.27,
  0.58,
  1.15,
  2.03,
  3.57,
  5,
  7,
  9.22,
  12.1,
  23.85],
 'remaining': [],
 'remaining_times': [],
 'dosing': [320],
 'dosing_type': ['oral'],
 'dosing_times': [0],
 'dosing_name': ['Theophylline'],
 'covariates': {'Weight': [80]}}